# SAR Intelligent Analysis - Batch Metrics Dashboard Let's explore the metrics and distributions for the 194 tested images using the inference results generated by our SAR Pipeline.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots beautiful
plt.style.use('ggplot')
sns.set_palette('deep')


In [ ]:
# Load JSON
with open('outputs/metrics/batch_results.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"Loaded {len(results)} image records.")

# Parse into DataFrames for easier plotting
records = []
for k, v in results.items():
    records.append({
        'image': v['image_name'],
        'num_objects': v['num_objects'],
        'terrain': v['scene']['terrain'],
        'terrain_conf': v['scene']['terrain_confidence'],
        'activity': v['scene']['activity'],
        'activity_conf': v['scene']['activity_confidence'],
        'priority_score': v['priority']['score'],
        'priority_tier': v['priority']['tier'],
    })
    
df = pd.DataFrame(records)
df.head()


## 1. Number of Objects Extracted per Image
A histogram depicting frequency of multi-object scenes.


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='num_objects', palette='viridis')
plt.title('Distribution of Object Detections (People/Animals) per Target Image')
plt.xlabel('Number of Objects in Image')
plt.ylabel('Count of Images')
plt.show()


## 2. Rescue Priority Visualization
Plotting Priority Scores spread and Priority Tiers distribution.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.histplot(df['priority_score'], bins=20, kde=True, color='red', ax=axes[0])
axes[0].set_title('Histogram of Priority Scores (0 - 100)')
axes[0].set_xlabel('Priority Score')

sns.countplot(data=df, x='priority_tier', order=['P1', 'P2', 'P3', 'P4'], palette='OrRd_r', ax=axes[1])
axes[1].set_title('Number of Images per Priority Tier')
axes[1].set_xlabel('Priority Tier')
axes[1].set_ylabel('Image Count')

plt.tight_layout()
plt.show()


## 3. Terrain and Activity Distributions
Visualizing the semantic environment of the search area.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))

# Terrain Pie
terrain_counts = df['terrain'].value_counts()
axes[0].pie(terrain_counts, labels=terrain_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
axes[0].set_title('Terrain Classification Across Dataset')

# Activity Counts
sns.countplot(data=df, y='activity', order=df['activity'].value_counts().index, palette='magma', ax=axes[1])
axes[1].set_title('Dominant Activities Detected')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
# Extract specifically what classes YOLO found
all_classes = []
for k, v in results.items():
    for d in v.get('detections', []):
        all_classes.append(d['label'])

plt.figure(figsize=(8,5))
sns.countplot(y=all_classes, order=pd.Series(all_classes).value_counts().index, palette='cubehelix')
plt.title('Instances of Different Objects Detected by YOLO')
plt.xlabel('Total Detected Instances')
plt.show()
